In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ─── Library Imports ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import math
import warnings
warnings.filterwarnings('ignore')

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13,
                     'axes.titleweight': 'bold', 'axes.labelsize': 11})

---
## 1. Dataset Initialization & Overview

We load the Hotel Booking dataset and examine its structure: dimensions, column names, data types, and a sample of raw rows.


In [ ]:
# ─── Load Data ─────────────────────────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/EDA final project/hotel_booking.csv')

print(f"Dataset shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print("-" * 50)
print("Column list:")
for col in df.columns:
    print(f"  • {col}")
print("-" * 50)
df.info()

In [ ]:
# ─── Preview 5 Rows ────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
df.head()

---
## 2. Data Dictionary

| Column | Description | Expected Type | Constraint |
|:---|:---|:---|:---|
| `hotel` | Hotel type | Categorical | Resort Hotel / City Hotel |
| `is_canceled` | Booking cancelled? | Boolean | 0 = No, 1 = Yes |
| `lead_time` | Days between booking & arrival | Integer | ≥ 0 |
| `arrival_date_year` | Year of arrival | Integer | 2015–2017 |
| `arrival_date_month` | Month of arrival | String | Jan–Dec |
| `arrival_date_week_number` | ISO week number | Integer | 1–53 |
| `arrival_date_day_of_month` | Day of arrival | Integer | 1–31 |
| `stays_in_weekend_nights` | Weekend nights booked | Integer | ≥ 0 |
| `stays_in_week_nights` | Weekday nights booked | Integer | ≥ 0 |
| `adults` | Number of adults | Integer | ≥ 0 |
| `children` | Number of children | Float→Int | ≥ 0 |
| `babies` | Number of babies | Integer | ≥ 0 |
| `meal` | Meal plan | Categorical | BB / HB / FB / No Meal |
| `country` | Guest country of origin | Categorical | ISO 3166 |
| `market_segment` | Market segment | Categorical | e.g., Online TA |
| `distribution_channel` | Booking channel | Categorical | e.g., Direct |
| `is_repeated_guest` | Returning guest? | Boolean | 0 / 1 |
| `previous_cancellations` | Prior cancellations by guest | Integer | ≥ 0 |
| `previous_bookings_not_canceled` | Prior kept bookings | Integer | ≥ 0 |
| `reserved_room_type` | Room type requested | Categorical | A–L |
| `assigned_room_type` | Room type assigned | Categorical | A–L |
| `booking_changes` | Number of changes made | Integer | ≥ 0 |
| `deposit_type` | Deposit type | Categorical | No Deposit / Non Refund / Refundable |
| `agent` | Travel agent ID | Float→Int | 0 = No Agent |
| `company` | Company ID | Float | High nulls → dropped |
| `days_in_waiting_list` | Days on waitlist | Integer | ≥ 0 |
| `customer_type` | Guest type | Categorical | Transient / Contract / etc. |
| `adr` | Average Daily Rate (€) | Float | ≥ 0 |
| `required_car_parking_spaces` | Parking requested | Integer | ≥ 0 |
| `total_of_special_requests` | Special requests count | Integer | ≥ 0 |
| `reservation_status` | Final booking status | Categorical | Canceled / Check-Out / No-Show |
| `reservation_status_date` | Date of last status change | Object→Date | Valid date |


---
## 3. Data Types Inspection

We audit each column's current dtype against its logical type to flag mismatches before cleaning.


In [ ]:
# ─── Data Types Audit ──────────────────────────────────────────────────────
print("Current Data Types:")
print(df.dtypes.to_string())
print("-" * 50)

issues = []
if df['reservation_status_date'].dtype == 'object':
    issues.append("  [ISSUE] 'reservation_status_date' is text  → convert to datetime")
for col in ['children', 'agent', 'company']:
    if col in df.columns and df[col].dtype == 'float64':
        issues.append(f"  [ISSUE] '{col}' is float (has NaNs)          → fill NaNs then cast to int")
for col in ['is_canceled', 'is_repeated_guest']:
    issues.append(f"  [NOTE]  '{col}' is int64 (0/1 flag)            → convert to bool")

print("\nIdentified Issues:")
for i in issues:
    print(i)

---
## 4. Missing Values Assessment

We quantify missing data per column before deciding on a handling strategy.


In [ ]:
# ─── Missing Values Report ─────────────────────────────────────────────────
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df)) * 100

missing_report = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct.round(2)
}).query('`Missing Count` > 0').sort_values('Missing Count', ascending=False)

print(f"Total rows in dataset: {len(df):,}")
print()
display(missing_report)

---
## 5. Missing Values Handling Plan

| Column | Strategy | Justification |
|:---|:---|:---|
| `company` | **Drop column** | > 90 % missing — adds noise, no analytical value |
| `agent` | **Fill with 0** | Null = no agent used; 0 preserves this business meaning |
| `country` | **Fill with "Unknown"** | < 1 % missing; preserves rows without distorting distribution |
| `children` | **Fill with 0** | Few missing; null implies zero children |


In [ ]:
# ─── Execute Missing Handling Plan ────────────────────────────────────────
df.drop(columns=['company'], inplace=True)
df['agent']    = df['agent'].fillna(0)
df['country']  = df['country'].fillna('Unknown')
df['children'] = df['children'].fillna(0)

# Verification
remaining = df[['agent', 'country', 'children']].isnull().sum()
print("Missing values after treatment:")
print(remaining.to_string())
print("\n✓ All targeted missing values resolved. 'company' column dropped.")

---
## 6. Data Type Corrections

With NaNs resolved, we safely cast each column to its logical type.


In [ ]:
# ─── Apply Type Corrections ────────────────────────────────────────────────
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])
df['children']                = df['children'].astype('int64')
df['agent']                   = df['agent'].astype('int64')
df['is_canceled']             = df['is_canceled'].astype(bool)
df['is_repeated_guest']       = df['is_repeated_guest'].astype(bool)

print("Updated types for corrected columns:")
cols_to_verify = ['reservation_status_date', 'children', 'agent',
                  'is_canceled', 'is_repeated_guest']
print(df[cols_to_verify].dtypes.to_string())
print("\n✓ All data type corrections applied.")

---
## 7. Duplicate Records Analysis

We first check for **exact duplicates** (identical across every column), then distinguish technical errors from legitimate repeat-guest bookings using a derived `Guest_ID`.


In [ ]:
# ─── Exact Duplicate Check ─────────────────────────────────────────────────
dup_count = df.duplicated().sum()
print(f"Exact duplicate rows: {dup_count}")

if dup_count > 0:
    print("\nSample duplicate rows:")
    display(df[df.duplicated()].head(5))
else:
    print("✓ No exact duplicate rows found.")

In [ ]:
# ─── Derived Guest_ID (Composite Key) ──────────────────────────────────────
df['Guest_ID'] = (df['name'].astype(str) + '|' +
                  df['email'].astype(str) + '|' +
                  df['phone-number'].astype(str))

total_dup_ids = df['Guest_ID'].duplicated().sum()

# Technical duplicates: same guest + same arrival date + same hotel
tech_errors = df.duplicated(
    subset=['Guest_ID', 'hotel',
            'arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month'],
    keep=False
).sum()

loyal_repeats = max(0, total_dup_ids - tech_errors)

print(f"Duplicate Guest IDs (repeat visitors)  : {total_dup_ids:,}")
print(f"Confirmed technical duplicates (errors): {tech_errors:,}")
print(f"Legitimate repeat bookings (loyalty)   : {loyal_repeats:,}")
print()
print("Decision: Keep all records — duplicate Guest_IDs represent returning customers,")
print("          not data entry errors.")

---
## 8. Validity Rules Check

We flag records that violate domain logic identified during the `describe()` review.

| Rule | Condition | Expected Action |
|:---|:---|:---|
| Financial integrity | `adr < 0` | Remove |
| Ghost bookings | adults + children + babies = 0 | Remove |
| Unrealistic occupancy | `adults > 10` | Remove |
| Negative lead time | `lead_time < 0` | Remove |


In [ ]:
# ─── Validity Violations Count ────────────────────────────────────────────
neg_adr       = df[df['adr'] < 0]
zero_guests   = df[(df['adults'] + df['children'] + df['babies']) == 0]
extreme_adults= df[df['adults'] > 10]
neg_lead      = df[df['lead_time'] < 0]

print(f"1. Negative ADR               : {len(neg_adr):,} rows")
print(f"2. Ghost bookings (0 guests)  : {len(zero_guests):,} rows")
print(f"3. Extreme adults (> 10)      : {len(extreme_adults):,} rows")
print(f"4. Negative lead time         : {len(neg_lead):,} rows")

if len(extreme_adults):
    print("\nSample — extreme adults:")
    display(extreme_adults[['name','adults','children','hotel','adr']].head(3))

if len(zero_guests):
    print("\nSample — ghost bookings:")
    display(zero_guests[['name','adults','children','babies','hotel']].head(3))

In [ ]:
# ─── Remove Invalid Records ────────────────────────────────────────────────
initial_size = len(df)

df = df[
    ((df['adults'] + df['children'] + df['babies']) > 0) &
    (df['adr'] >= 0) &
    (df['adults'] <= 10) &
    (df['lead_time'] >= 0)
].copy()

removed = initial_size - len(df)
print(f"Rows removed : {removed:,}")
print(f"Rows retained: {len(df):,}")
print()
print(f"Min ADR after cleaning      : {df['adr'].min():.2f}")
print(f"Min total guests after clean: {(df['adults']+df['children']+df['babies']).min()}")
print("✓ All impossible values removed.")

---
## 9. Category Cleanliness (Label Consistency)

We strip whitespace, unify casing, and map synonymous labels in key categorical columns to prevent fragmented frequency counts.


In [ ]:
# ─── Discover Raw Unique Values ────────────────────────────────────────────
cat_cols = ['hotel', 'meal', 'market_segment',
            'distribution_channel', 'reserved_room_type',
            'deposit_type', 'customer_type']

print("Raw unique values before cleaning:")
for col in cat_cols:
    print(f"  {col}: {sorted(df[col].unique())}")

In [ ]:
# ─── Apply Cleaning Plan ───────────────────────────────────────────────────
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Standardise meal labels (Undefined & SC both mean No Meal Plan)
meal_map = {
    'Undefined': 'No Meal', 'Sc': 'No Meal',
    'Bb': 'Bed & Breakfast', 'Hb': 'Half Board', 'Fb': 'Full Board'
}
df['meal'] = df['meal'].replace(meal_map)

print("Cleaned unique values for key columns:")
for col in ['meal', 'market_segment', 'deposit_type']:
    print(f"  {col}: {sorted(df[col].unique())}")
print("\n✓ Category labels cleaned.")

---
## 10. Comprehensive Numerical Summary

We compute mean, median, std, min, and max for all numerical columns, then highlight where mean ≠ median (evidence of skewness / outliers).


In [ ]:
# ─── Statistical Summary Table ─────────────────────────────────────────────
summary = df.describe().loc[['mean','50%','std','min','max']]
summary.rename(index={'50%': 'median'}, inplace=True)
print("Statistical Summary (transposed for readability):")
display(summary.T.round(2))

# ─── Mean vs Median Skewness Commentary ────────────────────────────────────
print("\nMean vs Median Observations (key columns):")
print("-" * 55)
for col in ['lead_time', 'adr', 'total_of_special_requests',
            'stays_in_week_nights', 'booking_changes']:
    if col not in df.columns:
        continue
    m, med = df[col].mean(), df[col].median()
    if med == 0:
        tag = "Extremely right-skewed (median = 0)"
    elif m > med * 1.15:
        tag = f"Right-skewed  (mean {m:.1f} > median {med:.1f})"
    elif m < med * 0.85:
        tag = f"Left-skewed   (mean {m:.1f} < median {med:.1f})"
    else:
        tag = f"Roughly symmetric (mean {m:.1f} ≈ median {med:.1f})"
    print(f"  {col:<35}: {tag}")

---
## 11. Numerical Distributions & Skewness Analysis

Histograms with KDE curves reveal the shape of each numerical variable.


In [ ]:
# ─── Distribution Histograms ───────────────────────────────────────────────
numeric_cols = df.select_dtypes(include='number').columns.tolist()
n_cols = 3
n_rows = math.ceil(len(numeric_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution: {col}')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.suptitle('Numerical Feature Distributions', y=1.01, fontsize=15, fontweight='bold')
plt.show()

# ─── Key Interpretations ────────────────────────────────────────────────────
interpretations = {
    'lead_time'               : 'Right-skewed — most guests book close to arrival; a long tail of early planners exists.',
    'adr'                     : 'Right-skewed — most rates cluster around the median; a few luxury prices pull the mean up.',
    'stays_in_week_nights'    : 'Right-skewed — short stays (1–4 nights) dominate; long stays are rare.',
    'adults'                  : 'Concentrated at 2 — reflects standard double-occupancy room bookings.',
    'total_of_special_requests': 'Highly right-skewed — the vast majority of guests make zero special requests.',
}
print("\nKey Distribution Insights:")
for col, text in interpretations.items():
    if col in numeric_cols:
        print(f"  • {col}: {text}")

---
## 12. Outlier Detection & Treatment

Boxplots expose extreme values. We decide per-column whether to **keep**, **cap (Winsorise)**, or **remove** outliers.

| Column | Decision | Rationale |
|:---|:---|:---|
| `adr` | **Cap at 99th percentile** | Extreme luxury prices distort the mean; the 99th cap preserves high-end reality |
| `lead_time` | **Keep** | Long advance bookings are legitimate business events |
| `stays_in_week_nights` | **Cap at 30 nights** | Stays > 30 nights are non-standard and likely mis-entered |
| `adults` | **Cap at ≤ 4** | > 4 adults in one room is unrealistic; already filtered above |
| `children`, `babies` | **Keep** | Large-family records are genuine |


In [ ]:
# ─── Boxplot Grid (Before Treatment) ──────────────────────────────────────
outlier_cols = ['lead_time', 'adr',
                'stays_in_weekend_nights', 'stays_in_week_nights', 'adults']

fig, axes = plt.subplots(1, len(outlier_cols),
                         figsize=(18, 5))
for ax, col in zip(axes, outlier_cols):
    sns.boxplot(y=df[col], ax=ax, color='salmon', width=0.5, flierprops=dict(marker='.', alpha=0.3))
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Outlier Detection — Boxplots (Before Treatment)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Apply Outlier Treatment ───────────────────────────────────────────────
# 1. Cap ADR at 99th percentile (Winsorise)
adr_99 = df['adr'].quantile(0.99)
df['adr'] = df['adr'].clip(upper=adr_99)

# 2. Remove extreme total stay durations
df = df[(df['stays_in_weekend_nights'] + df['stays_in_week_nights']) <= 30]

size_after = len(df)
print(f"ADR capped at        : {adr_99:.2f} €")
print(f"Rows after treatment : {size_after:,}")

# ─── Boxplot After Treatment ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(y=df['adr'], ax=axes[0], color='lightgreen',
            flierprops=dict(marker='.', alpha=0.3))
axes[0].set_title('ADR After Capping (99th pct)')

sns.boxplot(y=df['stays_in_week_nights'] + df['stays_in_weekend_nights'],
            ax=axes[1], color='lightblue',
            flierprops=dict(marker='.', alpha=0.3))
axes[1].set_title('Total Nights After Capping (≤30)')

plt.suptitle('Outlier Treatment Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 13. Categorical Variable Analysis

We display frequency tables and bar charts for the most analytically important categorical columns.


In [ ]:
# ─── Frequency Tables ──────────────────────────────────────────────────────
cat_cols_analysis = ['hotel', 'meal', 'market_segment',
                     'distribution_channel', 'deposit_type', 'customer_type']

for col in cat_cols_analysis:
    counts  = df[col].value_counts()
    pct     = df[col].value_counts(normalize=True).mul(100).round(2)
    tbl = pd.DataFrame({'Count': counts, 'Percentage (%)': pct})
    print(f"\n{'='*50}")
    print(f"  {col.upper()}")
    print('='*50)
    display(tbl)

In [ ]:
# ─── Bar Charts for Key Categorical Columns ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols_analysis):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=axes[i],
                  palette='Blues_r', saturation=0.85)
    axes[i].set_title(f'Frequency: {col}')
    axes[i].set_xlabel('Count')
    axes[i].set_ylabel('')
    # Add percentage labels
    total = len(df)
    for p in axes[i].patches:
        pct = f'{100 * p.get_width() / total:.1f}%'
        axes[i].annotate(pct,
                         (p.get_width() + total*0.002, p.get_y() + p.get_height()/2),
                         va='center', fontsize=9)

plt.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 14. Rare Category Handling

Categories with < 3 % frequency are consolidated into **'Other'** to avoid noise in models and visualisations.


In [ ]:
# ─── Identify & Group Rare Categories ─────────────────────────────────────
threshold = 0.03
cols_to_check = ['country', 'market_segment', 'distribution_channel', 'meal']

print("Rare Category Report (< 3 %):")
print("=" * 50)

for col in cols_to_check:
    rel_freq    = df[col].value_counts(normalize=True)
    rare_labels = rel_freq[rel_freq < threshold].index.tolist()
    if rare_labels:
        print(f"\n  {col.upper()} — {len(rare_labels)} rare labels found:")
        print(f"  Examples: {rare_labels[:5]}")
        df[col] = df[col].apply(lambda x: 'Other' if x in rare_labels else x)
        print(f"  → Grouped into 'Other' successfully.")
    else:
        print(f"\n  {col.upper()} — No rare categories found.")

print("\nUpdated COUNTRY top-10 after consolidation:")
display(df['country'].value_counts().head(10).to_frame('Count'))

---
## 15. Numerical Correlation Analysis

We compute the Pearson correlation matrix and highlight the five strongest pairwise relationships.


In [ ]:
# ─── Correlation Heatmap ───────────────────────────────────────────────────
numeric_df  = df.select_dtypes(include='number')
corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # show lower triangle only
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.4,
            annot_kws={'size': 8}, vmin=-1, vmax=1)
plt.title('Pearson Correlation Matrix (Lower Triangle)', fontsize=14)
plt.tight_layout()
plt.show()

# ─── Top 5 Strongest Pairs ─────────────────────────────────────────────────
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
top5  = upper.stack().reindex(
    upper.stack().abs().sort_values(ascending=False).index
).head(5)

print("\nTop 5 Strongest Numerical Relationships:")
print("-" * 55)
for (f1, f2), v in top5.items():
    direction = "positive" if v > 0 else "negative"
    print(f"  {f1}  ↔  {f2}")
    print(f"    Correlation: {v:.3f}  ({direction})")

---
## 16. Visualising Key Numerical Relationships

Scatter plots with regression lines confirm the nature of the two strongest correlations.


In [ ]:
# ─── Scatter Plots for Top Pairs ───────────────────────────────────────────
df_sample = df.sample(n=min(2000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Weekend vs week nights
sns.regplot(data=df_sample, x='stays_in_weekend_nights', y='stays_in_week_nights',
            ax=axes[0],
            scatter_kws={'alpha': 0.25, 'color': '#2c7bb6', 's': 15},
            line_kws={'color': '#d7191c', 'lw': 2})
axes[0].set_title('Weekend Nights vs Week Nights')
axes[0].set_xlabel('Weekend Nights')
axes[0].set_ylabel('Week Nights')

# Plot 2: Adults vs ADR
sns.regplot(data=df_sample, x='adults', y='adr',
            ax=axes[1],
            scatter_kws={'alpha': 0.25, 'color': '#1a9641', 's': 15},
            line_kws={'color': '#d7191c', 'lw': 2})
axes[1].set_title('Number of Adults vs ADR')
axes[1].set_xlabel('Adults')
axes[1].set_ylabel('Average Daily Rate (€)')

plt.suptitle('Key Numerical Relationships', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  1. Weekend vs Week Nights: Strong positive linear trend — longer stays are")
print("     proportionally distributed across the whole week.")
print("  2. Adults vs ADR: Upward slope confirms that room price increases with guest count.")

---
## 17. Numeric Effects by Category

We compare ADR across hotel types and market segments to understand revenue drivers.


In [ ]:
# ─── ADR by Hotel Type ─────────────────────────────────────────────────────
hotel_adr = (df.groupby('hotel')['adr']
               .agg(Mean='mean', Median='median', Std='std', Count='count')
               .reset_index()
               .rename(columns={'hotel': 'Hotel Type'}))

print("ADR by Hotel Type:")
display(hotel_adr.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.barplot(data=df, x='hotel', y='adr', estimator='mean',
            ax=axes[0], palette='Set2', capsize=0.1)
axes[0].set_title('Mean ADR by Hotel Type')
axes[0].set_ylabel('Mean ADR (€)')
axes[0].set_xlabel('')

# Violin plot — shows full distribution
sns.violinplot(data=df, x='hotel', y='adr',
               ax=axes[1], palette='Set2', inner='quartile')
axes[1].set_title('ADR Distribution by Hotel Type')
axes[1].set_ylabel('ADR (€)')
axes[1].set_xlabel('')

plt.suptitle('ADR Analysis by Hotel Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ─── ADR by Market Segment ──────────────────────────────────────────────────
seg_adr = (df.groupby('market_segment')['adr']
             .agg(Mean='mean', Count='count')
             .sort_values('Mean', ascending=False)
             .reset_index())

plt.figure(figsize=(12, 5))
sns.barplot(data=seg_adr, x='market_segment', y='Mean',
            palette='Blues_r', order=seg_adr['market_segment'])
plt.title('Mean ADR by Market Segment')
plt.ylabel('Mean ADR (€)')
plt.xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 18. Cancellation Drivers — Multivariate Analysis

We investigate three predictors of `is_canceled`: deposit type, lead time, and prior cancellations.


In [ ]:
# ─── Cancellation Dashboard ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Deposit Type vs Cancellation (stacked 100% bar)
dep_cancel = pd.crosstab(df['deposit_type'], df['is_canceled'],
                          normalize='index') * 100
dep_cancel.plot(kind='bar', stacked=True, ax=axes[0],
                color=['#2ca25f', '#de2d26'], edgecolor='white')
axes[0].set_title('Cancellation Rate by Deposit Type')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(['Not Canceled', 'Canceled'], loc='lower right')

# 2. Lead Time vs Cancellation (boxplot)
sns.boxplot(data=df, x='is_canceled', y='lead_time',
            ax=axes[1], palette='Set2')
axes[1].set_title('Lead Time vs Cancellation Status')
axes[1].set_xlabel('Canceled')
axes[1].set_ylabel('Lead Time (days)')
axes[1].set_xticklabels(['No (0)', 'Yes (1)'])

# 3. Avg Previous Cancellations (barplot)
sns.barplot(data=df, x='is_canceled', y='previous_cancellations',
            ax=axes[2], palette='coolwarm', capsize=0.1)
axes[2].set_title('Avg Prior Cancellations by Status')
axes[2].set_xlabel('Canceled')
axes[2].set_ylabel('Mean Prior Cancellations')
axes[2].set_xticklabels(['No (0)', 'Yes (1)'])

plt.suptitle('Cancellation Driver Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# ─── Summary Table ──────────────────────────────────────────────────────────
print("\nMean values by cancellation status:")
summary = df.groupby('is_canceled')[
    ['lead_time', 'previous_cancellations', 'adr', 'total_of_special_requests']
].mean().round(2)
summary.index = ['Not Canceled', 'Canceled']
display(summary)

---
## 19. Multivariate Correlation Heatmap

A final full-feature heatmap highlights multicollinearity and pinpoints features most correlated with `is_canceled`.


In [ ]:
# ─── Full Correlation Heatmap ──────────────────────────────────────────────
numeric_df2  = df.select_dtypes(include=np.number)
corr_matrix2 = numeric_df2.corr()

plt.figure(figsize=(15, 11))
sns.heatmap(corr_matrix2, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.4, center=0,
            annot_kws={'size': 7.5}, vmin=-1, vmax=1)
plt.title('Multivariate Correlation Heatmap — All Numeric Features',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ─── 3 Key Insights ─────────────────────────────────────────────────────────
print("Key Insights from the Heatmap:")
print()
print("  1. LEAD TIME ↔ IS_CANCELED (positive):")
print("     Bookings made far in advance have a notably higher cancellation rate.")
print("     This suggests guests are less committed when booking months ahead.")
print()
print("  2. WEEKEND NIGHTS ↔ WEEK NIGHTS (strong positive):")
print("     These two features are highly redundant — they measure the same")
print("     construct (total stay length). One could be dropped before modelling.")
print()
print("  3. ADR ↔ ADULTS / CHILDREN (moderate positive):")
print("     Room pricing is primarily driven by guest count rather than")
print("     booking channel, confirming a per-person pricing model.")

---
## 20. Final EDA Summary: Insights, Risks & Roadmap

### I. Top 5 Analytical Insights

1. **Lead Time → Cancellation**: Significant positive correlation. Guests booking months in advance are substantially more likely to cancel — a key risk signal for revenue management.
2. **Deposit Type Predictive Power**: "Non-Refund" deposit holders paradoxically show a high cancellation rate, likely driven by the OTA (Online Travel Agency) segment behaviour.
3. **Seasonality**: Booking volume peaks in summer (July/August), especially for Resort Hotels. Temporal features carry strong predictive information.
4. **Customer Loyalty**: Repeat guests (`is_repeated_guest = True`) cancel at a significantly lower rate — a high-value binary feature for cancellation prediction.
5. **Market Segment Volatility**: "Online TA" dominates volume (~47 %) but carries the highest cancellation risk, making segment a strong categorical predictor.

---

### II. Top 5 Data Quality Risks Identified & Resolved

| # | Issue | Status |
|:---|:---|:---|
| 1 | Missing values in `agent`, `country`, `children`, `company` | ✓ Resolved (imputed / dropped) |
| 2 | ADR outliers (extreme luxury rates) | ✓ Capped at 99th percentile |
| 3 | Ghost bookings (0 total guests) | ✓ Removed |
| 4 | Inconsistent categorical labels (e.g., `meal` = 'Sc' / 'Undefined') | ✓ Standardised |
| 5 | Rare country / segment categories cluttering distributions | ✓ Consolidated into 'Other' |

---

### III. Recommended Next Steps

1. **Feature Engineering**: Create `total_stay` (weekend + week nights), `total_guests` (adults + children + babies), and an `arrival_season` label.
2. **Encoding**: One-hot encode low-cardinality categoricals; target-encode `country`.
3. **Scaling**: Standardise `lead_time`, `adr`, and `days_in_waiting_list` before modelling.
4. **Modelling**: Train a classification model (Logistic Regression → Random Forest → XGBoost) on `is_canceled` as the target variable.
5. **Orange Data Mining**: Import the cleaned CSV into Orange and replicate key visualisations (distributions, scatter plots, heatmap) as required by the project rubric.
